# Predictive Maintenance & Equipment Failure Prediction System
### Academic & Industrial Machine Learning Project
---
**Objective:** Build an end-to-end Machine Learning system to analyze industrial sensor telemetry, predict machine breakdown probability, classify equipment risk, and recommend proactive maintenance.

## 1. Imports & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Load AI4I 2020 Predictive Maintenance Dataset
df = pd.read_csv('../data/predictive_maintenance.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis & Failure Breakdown

In [ ]:
print("Target Distribution (Machine Failure):")
print(df['Machine failure'].value_counts(normalize=True) * 100)

failure_modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
print("\nFailure Mode Breakdown:")
for fm in failure_modes:
    print(f"{fm}: {df[fm].sum()} occurrences")

## 3. Domain Feature Engineering
We engineer four physics-based variables:
1. **Temperature Difference:** $\Delta T = T_{process} - T_{air}$
2. **Mechanical Power (kW):** $P = \tau \cdot \omega = \text{Torque} \cdot \left(\frac{2\pi N}{60}\right) / 1000$
3. **Overstrain Product:** $\text{Tool wear} \times \text{Torque}$
4. **Thermal Expansion Ratio:** $T_{process} / T_{air}$

In [ ]:
def engineer_features(data):
    d = data.copy()
    d['Temp difference [K]'] = d['Process temperature [K]'] - d['Air temperature [K]']
    d['Power [kW]'] = d['Rotational speed [rpm]'] * d['Torque [Nm]'] * (2 * np.pi / 60.0) / 1000.0
    d['Overstrain factor [min x Nm]'] = d['Tool wear [min]'] * d['Torque [Nm]']
    d['Temp ratio'] = d['Process temperature [K]'] / d['Air temperature [K]']
    return d

df_feat = engineer_features(df)
df_feat[['Temp difference [K]', 'Power [kW]', 'Overstrain factor [min x Nm]', 'Temp ratio']].describe()

## 4. Model Training, Evaluation & Benchmarking

In [ ]:
raw_numeric = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
eng_numeric = ['Temp difference [K]', 'Power [kW]', 'Overstrain factor [min x Nm]', 'Temp ratio']
num_cols = raw_numeric + eng_numeric
cat_cols = ['Type']

X = df_feat[num_cols + cat_cols]
y = df_feat['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

neg_cnt = (y_train == 0).sum()
pos_cnt = (y_train == 1).sum()
scale_weight = neg_cnt / pos_cnt

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced', random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.08, scale_pos_weight=scale_weight, eval_metric='logloss', random_state=42)
}

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}\n")